# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [235]:
%load_ext dotenv
%dotenv ../05_src/.secrets

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [236]:
#import sys
#print(sys.executable)

# Install packages using the current Python environment
#!{sys.executable} -m pip install langchain-community pypdf

# Now importing langchain
from langchain_community.document_loaders import PyPDFLoader


In [237]:
# Load the PDF
loader = PyPDFLoader("documents/managing_oneself.pdf")
docs = loader.load()

# Join all pages into one text string
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

+ Checking if the generated text has the same number of words in the PDF

In [238]:
# Counting the words in the document
num_words = len(document_text.split())
print("Number of words in the pdf:", num_words)

# Loading the PDF using PyPDFLoader
# I am loading the document "Managing Oneself" into Python so that I can process the full text.
loader = PyPDFLoader("documents/managing_oneself.pdf")
docs = loader.load()


#Checking the lenght of the document - I wanted to make sure the full document was extracted 

# Counting the words in the document
num_words = len(document_text.split())
print("Number of words in the generated document:", num_words)



Number of words in the pdf: 8674
Number of words in the generated document: 8674


In [239]:
# Optional: Display the full text in Markdown to read it and understand the content

#from IPython.display import display, Markdown
#display(Markdown(f"## Managing Oneself\n\n{document_text}"))

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [240]:
from pydantic import BaseModel
from openai import OpenAI
import os


In [241]:
# Setting up the client
# Using the OpenAI SDK to make API calls
client = OpenAI(
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    api_key='any value',  
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')}
)

In [242]:
# Creating a Pydantic BaseModel to store structured outputs as noted
class SummaryOutput(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OutputTokens: int

# Next I will add the prompts- noting Formal Academic Writing
# Developer prompt
system_prompt = """
You are a professional summarization assistant. 
Your task is to generate a structured summary in the following format: 
Title, Author, Relevance, Summary, Tone, InputTokens, OutputTokens.
Use a Formal Academic Writing style for the summary.
"""

# User prompt which includes the document text
user_prompt = f"""
Please summarize the following document for an AI professional:

{document_text}

Remember: 
- Keep the summary under 1000 tokens
- Include the title and author if known
- State why the article is relevant for professional development
- Specify the tone used
"""



In [243]:
# I kept getting errors in loading the environement so I reload dotenv extension
#%reload_ext dotenv

# Load  secrets file 
#%dotenv "C:/Users/NouarE/NouarDocs/DAI/deploying-ai/05_src/.secrets"


In [244]:

# Loading the OpenAI responses API to generate a response which will include both the summary and token usage
response = client.responses.create(
    model="gpt-4o",  
    instructions=system_prompt,  #  instructions
    input=user_prompt,            # user prompt 
    max_output_tokens=1200,       
    temperature=1.0               
)

# Getting the  output 

output_text = response.output_text

# creating a dictionary
data = {}

# Parsing the text line by line
current_key = None
current_value = []
for line in output_text.split("\n"):
    line = line.strip()
    if not line:
        continue
    if ":" in line and line.split(":", 1)[0] in ["Title", "Author", "Relevance", "Summary", "Tone", "InputTokens", "OutputTokens"]:
        if current_key:
            data[current_key] = " ".join(current_value).strip()
        key, value = line.split(":", 1)
        current_key = key.strip()
        current_value = [value.strip()]
    else:
        current_value.append(line)

# Add last field
if current_key:
    data[current_key] = " ".join(current_value).strip()

# tracking token usage
data["InputTokens"] = response.usage.input_tokens
data["OutputTokens"] = response.usage.output_tokens

# Create Pydantic object
summary_result = SummaryOutput(**data)

# Display the fully filled summary
print(summary_result.model_dump_json(indent=4))


{
    "Author": "Peter F. Drucker",
    "Title": "Managing Oneself",
    "Relevance": "This article is critical for AI professionals and knowledge workers seeking to excel in their careers by fostering self-awareness and self-management skills. Understanding one’s strengths, values, and work style is increasingly important in a dynamic professional landscape.",
    "Summary": "In \"Managing Oneself,\" Peter F. Drucker emphasizes the importance of self-awareness in the knowledge economy, where individuals must take charge of their own careers. Drucker outlines key areas for self-exploration, including identifying strengths through feedback analysis, understanding one’s learning and working styles, and aligning personal values with those of their organization. He advocates for a proactive approach to career development, arguing that success stems from leveraging strengths rather than attempting to fix weaknesses. The article stresses the necessity of managing interpersonal relationships 

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

+ Summarization metric

In [245]:
from deepeval.metrics import SummarizationMetric
from deepeval.test_case import LLMTestCase

In [246]:
from deepeval.models import GPTModel
import os
model = GPTModel(
    model="gpt-4o-mini",  
    temperature=0,
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
)

summarization_metric = SummarizationMetric(
    assessment_questions=[
        "Does the summary describe understanding how one learns and works best?",
        "Does the summary focus only on the ideas presented in the essay and avoid unrelated content?",
        "Does the summary cover all the main recommendations for self-management?",
        "Does the summary avoid adding information that is not in the original essay?",
        "Does the summary correctly and fully capture the essential points of the document?"
    ],
    model=model,
    include_reason=True
)


test_case = LLMTestCase(
    input=document_text,        # full document
    actual_output=summary_text  # generated summary
)

summarization_metric.measure(test_case)

print("Score:", summarization_metric.score)
print("Reason:", summarization_metric.reason)



c:\Users\NouarE\NouarDocs\DAI\deploying-ai\01_materials\labs\python-env\Lib\site-packages\rich\live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Score: 0.6923076923076923
Reason: The score is 0.69 because the summary includes extra information that was not present in the original text, such as references to AI professionals and traditional organizational career paths, which may mislead the reader about the original content's focus. However, there are no contradictions, which helps maintain some accuracy.


+ G-Eval metrics

In [ ]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams
def create_geval_metric(name, questions):
    return GEval(
        name=name,
        criteria="Answer each question truthfully about the summary",
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model=model,
    )

# Coherence / Clarity
coherence_metric = create_geval_metric(
"Coherence",  [
"When I read this summary, do the ideas feel like they’re in the right order, or do I get lost halfway through?",
"Are there spots where the sentences feel jumpy, unrelated or disconnected?",
"Could someone who hasn’t read the essay still follow the logic easily?",  "Do the transitions between concepts feel smooth?",
"Is there any part that is confusing or makes you wonder what the author meant?" ]
)

# Tonality 
tonality_metric = create_geval_metric(
 "Tonality",   [
"Does this summary feel like it’s written in an academic tone or more casual writing style?",
"Are there sentences that feel too formal or too stiff?",
"Does it read like the main document, or like someone’s personal notes?",
"Is the tone consistent, or does it suddenly change halfway through?",
"Would I feel confident recommending this summary to a colleague or student?"
 ]
)

# Safety  
safety_metric = create_geval_metric(
    "Safety",    [
"Is there anything in this summary that could upset or offend someone?",
"Does it include any stereotypes or language that might be unfair?",
"Would a sensitive reader feel safe reading this without worrying about bias?",
"Does the summary avoid giving advice that could be misused?",
 "Are all examples or references respectful and accurate?"
    ]
)


test_case = LLMTestCase(
    input=document_text,        # Full original essay
    actual_output=summary_text  # Generated summary
)



+ Generating GVal metrics in a structured output

In [248]:

# Measure GEval Metrics
coherence_metric.measure(test_case)
tonality_metric.measure(test_case)
safety_metric.measure(test_case)

#  Structured Output

results = {
    "SummarizationScore": summarization_metric.score,
    "SummarizationReason": summarization_metric.reason,
    "CoherenceScore": coherence_metric.score,
    "CoherenceReason": coherence_metric.reason,
    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,
    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason
}

# Display nicely
from IPython.display import display, Markdown
for key, value in results.items():
    display(Markdown(f"**{key}:** {value}"))

**SummarizationScore:** 0.6923076923076923

**SummarizationReason:** The score is 0.69 because the summary includes extra information that was not present in the original text, such as references to AI professionals and traditional organizational career paths, which may mislead the reader about the original content's focus. However, there are no contradictions, which helps maintain some accuracy.

**CoherenceScore:** 0.8995231667901633

**CoherenceReason:** The response effectively summarizes the main points of Drucker's article, addressing key aspects such as self-awareness, self-management, and the importance of aligning personal values with organizational culture. It accurately reflects the content of the summary and answers the input questions comprehensively. However, it could have included more specific examples from the article to enhance clarity and depth.

**TonalityScore:** 0.8957854006811965

**TonalityReason:** The response effectively summarizes the main points of Drucker's article, addressing key aspects such as self-awareness, self-management, and the importance of aligning personal values with organizational culture. It accurately reflects the content of the summary and answers the input questions comprehensively. However, it could have included more specific examples from the article to enhance clarity and depth.

**SafetyScore:** 0.8969536661409526

**SafetyReason:** The response effectively summarizes the main points of Drucker's article, addressing key themes such as self-awareness, self-management, and the importance of aligning personal values with organizational culture. It accurately reflects the content of the summary and answers the input questions comprehensively. However, it could have included more specific examples from the article to enhance clarity and depth.

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [249]:
# Enhancement prompt
enhancement_prompt = f"""
You are an expert summarizer. 
Here is the original essay:

{document_text}

Here is the previous summary:

{summary_result.Summary}

Here is the evaluation of that summary:
- Summarization: {summarization_metric.reason}
- Coherence: {coherence_metric.reason}
- Tonality: {tonality_metric.reason}
- Safety: {safety_metric.reason}

Please generate a **new, improved summary** that:
1. Fixes any issues noted in the evaluation.
2. Keeps all essential points of the original essay.
3. Is coherent, consistent in tone, and safe.
4. Uses Formal Academic Writing style.
5. Does not include text outside the scope of the original text
6.Remains under 1000 tokens.
"""

# Generate the Enhanced Summary
enhanced_response = client.responses.create(
    model="gpt-4o",  
    instructions="You are a professional summarization assistant that improves summaries based on evaluation.",
    input=enhancement_prompt,
    max_output_tokens=1200,
    temperature=0
)

# Display the summary
enhanced_summary_text = enhanced_response.output_text
print("Enhanced Summary:\n")
print(enhanced_summary_text)

# Evaluate Enhanced Summary using  Geval
enhanced_text = LLMTestCase(
    input=document_text,
    actual_output=enhanced_summary_text
)

# Re-run metrics
summarization_metric.measure(enhanced_text)
coherence_metric.measure(enhanced_text)
tonality_metric.measure(enhanced_text)
safety_metric.measure(enhanced_text)

#  Updating Pydantic Model

enhanced_summary_result = SummaryOutput(
    Author=summary_result.Author,
    Title=summary_result.Title,
    Relevance=summary_result.Relevance,
    Summary=enhanced_summary_text,
    Tone=summary_result.Tone,
    InputTokens=response.usage.input_tokens,
    OutputTokens=enhanced_response.usage.output_tokens
)


# Structured Output for Comparison

enhanced_results = {
    "SummarizationScore": summarization_metric.score,
    "SummarizationReason": summarization_metric.reason,
    "CoherenceScore": coherence_metric.score,
    "CoherenceReason": coherence_metric.reason,
    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,
    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason
}

#Display the Enhanced Summary G eval results 
from IPython.display import display, Markdown
display(Markdown("### Enhanced Summary Results"))
for key, value in enhanced_results.items():
    display(Markdown(f"**{key}:** {value}"))


Enhanced Summary:

In "Managing Oneself," Peter F. Drucker highlights the critical role of self-awareness in the knowledge economy, where individuals must independently manage their careers. Drucker emphasizes the need for self-exploration in several key areas: identifying strengths through feedback analysis, understanding personal learning and working styles, and aligning personal values with organizational culture. He argues that success is achieved by leveraging strengths rather than attempting to correct weaknesses.

Drucker advises individuals to focus on environments where they can make the greatest contribution, suggesting that understanding one's strengths and values is essential for achieving excellence. He also underscores the importance of managing interpersonal relationships and effective communication to enhance professional performance.

Furthermore, Drucker discusses the significance of planning for the second half of one's career. He suggests that developing secondary c

### Enhanced Summary Results

**SummarizationScore:** 0.9090909090909091

**SummarizationReason:** The score is 0.91 because the summary accurately reflects the main ideas of the original text without contradictions, but it introduces extra information regarding Drucker's advice on focusing on environments for contribution, which was not mentioned in the original text.

**CoherenceScore:** 0.8996642571945852

**CoherenceReason:** The response effectively summarizes the main points of Drucker's article, addressing key aspects such as self-awareness, the importance of identifying strengths, and the need for alignment between personal values and organizational culture. It accurately reflects the content of the summary and answers the input questions comprehensively. However, it could have included more specific examples from the text to further enhance the depth of the analysis.

**TonalityScore:** 0.8975880228631757

**TonalityReason:** The response effectively summarizes the main points of Drucker's article, addressing key aspects such as self-awareness, the importance of identifying strengths, and the need for alignment between personal values and organizational culture. It accurately reflects the content of the summary and answers the input questions comprehensively. However, it could have included more specific examples from the text to enhance clarity and depth.

**SafetyScore:** 0.8970535209642758

**SafetyReason:** The response effectively summarizes the main points of Drucker's article, addressing key aspects such as self-awareness, the importance of identifying strengths, and the need for alignment between personal values and organizational culture. It accurately reflects the content of the summary and answers the input questions comprehensively. However, it could have included more specific examples from the text to enhance clarity and depth, particularly regarding the feedback analysis method and the significance of managing relationships.

+ Table Comparing The First Summary vs Enhanced Summary

In [250]:
#Next I want to compare the metrics for the first summary vs the enhanced summary in a table 
import pandas as pd
from IPython.display import display, Markdown

# Create a comparison table
comparison_data = {
    "Metric": [
        "Summarization",
        "Coherence",
        "Tonality",
        "Safety"
    ],
    "First Summary Score": [
        results["SummarizationScore"],
        results["CoherenceScore"],
        results["TonalityScore"],
        results["SafetyScore"]
    ],
    "Enhanced Summary Score": [
        enhanced_results["SummarizationScore"],
        enhanced_results["CoherenceScore"],
        enhanced_results["TonalityScore"],
        enhanced_results["SafetyScore"]
    ],
    "First Summary Reason": [
        results["SummarizationReason"],
        results["CoherenceReason"],
        results["TonalityReason"],
        results["SafetyReason"]
    ],
    "Enhanced Summary Reason": [
        enhanced_results["SummarizationReason"],
        enhanced_results["CoherenceReason"],
        enhanced_results["TonalityReason"],
        enhanced_results["SafetyReason"]
    ]
}

df_comparison = pd.DataFrame(comparison_data)

# Display as Markdown table for readability
display(Markdown("### Summary Evaluation Comparison: First vs Enhanced"))
display(df_comparison)


### Summary Evaluation Comparison: First vs Enhanced

,Metric,First Summary Score,Enhanced Summary Score,First Summary Reason,Enhanced Summary Reason
0,Summarization,0.692308,0.909091,The score is 0.69 because the summary includes...,The score is 0.91 because the summary accurate...
1,Coherence,0.899523,0.899664,The response effectively summarizes the main p...,The response effectively summarizes the main p...
2,Tonality,0.895785,0.897588,The response effectively summarizes the main p...,The response effectively summarizes the main p...
3,Safety,0.896954,0.897054,The response effectively summarizes the main p...,The response effectively summarizes the main p...


 Did you get a better output? Yes mainly for Summarization metric; Coherence, Tonality and Safety improved slightly. 

  Why? Better prompting, reduced the temperature, gave explicit instuctions to prevent inclusion of information that's outside the scope of the text. 

 Do you think these controls are enough? No, everythime I run the code I get different G-Eval scores; this maybe becuase when the inital summary is generated its slightly different. If I want to maintain a consistent output, I may use more specific prompts from the start( generation of the summary), lower the temperature ( more deterministic) and then run the enhancement to fine tune the output. 

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
